In [54]:
import pandas as pd
from pathlib import Path

#Carga de dataframes Enero y Febrero
ruta = Path.home() / "Downloads" / "dataset_final_Enero.parquet"
df1 = pd.read_parquet(ruta)
ruta = Path.home() / "Downloads" / "dataset_final_Febrero.parquet"
df2 = pd.read_parquet(ruta)


#Unión de los dos dataframes
df = pd.concat([df1,df2])

df

,date,match_key,stop_id,route_id,direction,delay_seconds,lagged_delay_1,lagged_delay_2,route_rolling_delay,actual_headway_seconds,...,afecta_durante,afecta_despues,category,num_updates,timestamp_start,seconds_since_last_alert,is_alert_just_published,seconds_to_next_alert,alert_in_next_15m,alert_in_next_30m
0,2025-01-01,137350_D..S07R,B12S,D,S,-30.0,NaN,NaN,NaN,NaN,...,0,0,NaN,0.0,NaT,NaN,0,9840.0,0,0
1,2025-01-01,287600_3..N01R,254N,NaN,N,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,0.0,NaT,NaN,0,NaN,0,0
2,2025-01-01,284850_1..S03R,123S,NaN,S,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,0.0,NaT,NaN,0,NaN,0,0
3,2025-01-01,141700_J..N40R,J30N,J,N,-58.0,NaN,NaN,92.0,NaN,...,0,0,NaN,0.0,NaT,NaN,0,1018.0,0,1
4,2025-01-01,140200_J..N40R,J20N,J,N,92.0,NaN,NaN,NaN,NaN,...,0,0,NaN,0.0,NaT,NaN,0,1018.0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5683203,2025-02-28,284000_1..S03R,139S,NaN,S,NaN,NaN,NaN,NaN,56.0,...,0,0,NaN,0.0,NaT,NaN,0,NaN,0,0
5683204,2025-02-28,134000_D..S05R,D43S,D,S,57.0,162.0,57.0,53.0,247.0,...,0,0,Delay,0.0,2025-02-27 16:58:00,111717.0,0,NaN,0,0
5683205,2025-02-28,286000_2..S01R,214S,NaN,S,NaN,NaN,NaN,NaN,933.0,...,0,0,NaN,0.0,NaT,NaN,0,NaN,0,0
5683206,2025-02-28,286750_3..S88R,120S,NaN,S,NaN,NaN,NaN,NaN,45.0,...,0,0,NaN,0.0,NaT,NaN,0,NaN,0,0


In [ ]:
#Ordenamos por línea, parada y timestamp
df = df.sort_values(['route_id', 'stop_id', 'merge_time'])
grupo = ['route_id', 'stop_id']

#Creación de nuevas variables de delay para poder ampliar la ventana de observaciones a 30 minutos aproximadamente
df['lagged_delay_3'] = df.groupby(grupo)['delay_seconds'].shift(3)
df['lagged_delay_4'] = df.groupby(grupo)['delay_seconds'].shift(4)

#Frecuencia de las observaciones
frecuencia = (
    df.groupby(grupo)['merge_time']
      .diff()
      .dt.total_seconds()
      .div(60)          # convertir a minutos
      .median()
)

print(f"Frecuencia mediana entre observaciones: {frecuencia:.1f} minutos")
PASOS_FUTURO = round(15/frecuencia) #Steps que hay que dar para crear el gap

Frecuencia mediana entre observaciones: 8.6 minutos


In [56]:
df['alerta_15min'] = (
    df.groupby(grupo)['is_alert_just_published']    # columna de alerta
      .shift(-PASOS_FUTURO)
)

#Eliminamos nulos
df = df.dropna(subset=['alerta_15min'])
df['alerta_15min'] = df['alerta_15min'].astype(int)

In [57]:
#Seleccion de variables
features = [
    'delay_seconds', 'lagged_delay_1', 'lagged_delay_2',
    'route_rolling_delay', 'actual_headway_seconds',
    'is_unscheduled', 'num_updates', 'scheduled_time_to_end',
    'stops_to_end', 'route_id', 'direction',
    'hour_sin', 'hour_cos', 'dow', 'is_weekend',
    'n_eventos_afectando', 'tipo_referente',
    'afecta_previo', 'afecta_durante', 'afecta_despues',
    'temp_extreme',
]
target = 'alerta_15min'


In [58]:
df_sorted = df.sort_values('merge_time')

#Division X e y
X = df_sorted[features]
y = df_sorted[target]

print(f"Features: {len(features)}")
print(f"Filas:    {len(X):,}")
print(f"\nDistribución del target:")
print(y.value_counts(normalize=True).round(3))


#División de los datos en Entrenamiento-Validación-Test
n        = len(df_sorted)
i_train  = int(n * 0.70)
i_val    = int(n * 0.85)

X_train, y_train = X.iloc[:i_train], y.iloc[:i_train]
X_val, y_val = X.iloc[i_train:i_val], y.iloc[i_train:i_val]
X_test, y_test = X.iloc[i_val:], y.iloc[i_val:]

print(f"Train: {len(X_train):,}  ({len(X_train)/n*100:.0f}%)")
print(f"Val:   {len(X_val):,}  ({len(X_val)/n*100:.0f}%)")
print(f"Test:  {len(X_test):,}  ({len(X_test)/n*100:.0f}%)")

Features: 21
Filas:    8,938,771

Distribución del target:
alerta_15min
0    0.992
1    0.008
Name: proportion, dtype: float64
Train: 6,257,139  (70%)
Val:   1,340,816  (15%)
Test:  1,340,816  (15%)
